# kor-minish: BAAI/bge-m3 → 한글 정적 임베딩 distill (Colab T4)

## 사용법
1. **Runtime → Change runtime type → T4 GPU** 선택
2. 위에서부터 셀 순서대로 실행
3. vocab 업로드 셀에서 로컬의 `vocab_ko.txt` 선택
4. distill 완료 후 자동으로 zip 다운로드

예상 소요: 5~10분 (vocab 30K, PCA 256 기준)

In [ ]:
# 1) GPU 확인
!nvidia-smi

In [ ]:
# 2) 의존성 설치
!pip install -q 'model2vec[distill]>=0.6.0' 'sentence-transformers>=3.0'

In [ ]:
# 3) torch가 CUDA를 잡았는지 확인
import torch
print('cuda:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

In [ ]:
# 4) vocab_ko.txt 업로드 (로컬에서 만든 파일 선택)
from google.colab import files
uploaded = files.upload()
vocab_path = next(iter(uploaded.keys()))
with open(vocab_path, encoding='utf-8') as f:
    vocab = [w.strip() for w in f if w.strip()]
print(f'vocab size: {len(vocab):,}')
print('preview:', vocab[:10])

In [ ]:
# 5) Distill (T4에서 ~5분)
from model2vec.distill import distill

BASE_MODEL = 'BAAI/bge-m3'
OUT_DIR = 'kor-minish-bge-m3-ko'
PCA_DIMS = 256  # 0 또는 None으로 두면 원본 1024 유지

m2v = distill(
    model_name=BASE_MODEL,
    vocabulary=vocab,
    pca_dims=PCA_DIMS,
    apply_zipf=True,
    device='cuda',
)
m2v.save_pretrained(OUT_DIR)
print(f'saved -> {OUT_DIR}')

In [ ]:
# 6) Quick sanity check (T4 위에서 한국어 문장쌍 코사인)
import numpy as np
from model2vec import StaticModel

m = StaticModel.from_pretrained(OUT_DIR)

def cos(a, b):
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

pairs = [
    ('강아지가 공원에서 뛰어논다', '개가 공원에서 놀고 있다'),
    ('강아지가 공원에서 뛰어논다', '주식 시장이 폭락했다'),
    ('인공지능 모델 학습', 'AI 모델을 훈련시킨다'),
    ('김치찌개 레시피', '된장찌개 만드는 법'),
    ('김치찌개 레시피', '자동차 엔진 정비'),
]
sents = [s for p in pairs for s in p]
vecs = m.encode(sents)
print(f'dim = {vecs.shape[1]}')
for i, (a, b) in enumerate(pairs):
    print(f'{cos(vecs[2*i], vecs[2*i+1]):+.3f}  {a}  <->  {b}')

In [ ]:
# 7) 결과 압축 + 다운로드
import shutil
zip_path = shutil.make_archive(OUT_DIR, 'zip', OUT_DIR)
print(f'zip: {zip_path}')
files.download(zip_path)